## Phase 0: Configuration and Field Mapping

In [165]:
### CONFIGURATION & DEMO VARIABLES ###

# Set the path for your input CSV files
loinc_csv_path = 'SmallTestCSVs/Loinc.csv'
part_link_csv_path = 'SmallTestCSVs/Part.csv'
answer_list_csv_path = 'SmallTestCSVs/AnswerList.csv'
linguistic_variants_path = 'SmallTestCSVs/LinguisticVariants'
panels_and_forms_csv_path = 'SmallTestCSVs/PanelsAndForms.csv'
loinc_answer_list_link_csv_path = 'SmallTestCSVs/LoincAnswerListLink.csv'
map_to_csv_path = 'SmallTestCSVs/MapTo.csv'
component_hierarchy_file_path = "SmallTestCSVs/ComponentHierarchyBySystem.csv"


# Set the output path for the transformed JSONL file
output_folder = 'output'

# Set this to 1 or 2 to run the notebook with example data instead of your real CSVs.
# NOTE: The demo data below is for illustration; real-world data is much larger.
mode = 0  # 0 = full run with all data, 1 = test_mode with real subset of data up to 15 records per CSV, 2 = demo_mode with example data

In [166]:
### PACKAGE IMPORTS ###

# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations, often used for NaN values
import numpy as np

# Import json for saving to JSON Lines format
import json

#Import StringIO to handle in-memory text streams
from io import StringIO

# Import os for file and directory operations
import os

# Import re for regular expression operations
import re

In [167]:
# Example DataFrames for demo_mode (mode = 2)
# These are based on the CSV snippets you provided.
demo_loinc_df = pd.DataFrame({
    'LOINC_NUM': ['100000-9', '100001-7'],
    'COMPONENT': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'PROPERTY': ['Hx', 'LP431396-3'],
    'TIME_ASPCT': ['Pt', 'Pt'],
    'SYSTEM': ['^Patient', 'Ser'],
    'SCALE_TYP': ['Nar', 'Qn'],
    'SHORTNAME': ['Health Info Pioneer+Father of LOINC', 'Health Info Pioneer+Cofounder of LOINC'],
    'LONG_COMMON_NAME': ['Health informatics pioneer and the father of LOINC', 'Health informatics pioneer and cofounder of LOINC'],
    'STATUS': ['ACTIVE', 'ACTIVE']
})

demo_part_link_df = pd.DataFrame({
    'LoincNumber': ['100000-9', '100000-9', '100000-9', '100000-9', '100000-9'],
    'LongCommonName': ['Health informatics pioneer and the father of LOINC'] * 5,
    'PartNumber': ['LP431397-1', 'LP6817-3', 'LP6960-1', 'LP310005-6', 'LP7749-7'],
    'PartName': ['Health informatics pioneer and the father of LOINC', 'Hx', 'Pt', '^Patient', 'Nar'],
    'PartCodeSystem': ['http://loinc.org'] * 5,
    'PartTypeName': ['COMPONENT', 'PROPERTY', 'TIME', 'SYSTEM', 'SCALE'],
    'LinkTypeName': ['Primary'] * 5,
    'Property': ['http://loinc.org/property/COMPONENT', 'http://loinc.org/property/PROPERTY', 'http://loinc.org/property/TIME_ASPCT', 'http://loinc.org/property/SYSTEM', 'http://loinc.org/property/SCALE_TYP']
})

demo_answer_list_df = pd.DataFrame({
    'AnswerListId': ['LL1000-0', 'LL1000-0', 'LL1000-0', 'LL1001-8'],
    'AnswerListName': ['PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_13_30D bread amt', 'PhenX05_14_30D freq amts'],
    'AnswerListOID': ['1.3.6.1.4.1.12009.10.1.165'] * 3 + ['1.3.6.1.4.1.12009.10.1.166'],
    'ExtDefinedYN': ['N'] * 4,
    'AnswerStringId': ['LA13825-7', 'LA13838-0', 'LA13892-7', 'LA6270-8'],
    'SequenceNumber': [1, 2, 3, 1],
    'DisplayText': ['1 slice or 1 dinner roll', '2 slices or 2 dinner rolls', 'More than 2 slices or 2 dinner rolls', 'Never']
})

# Linguistic variant CSV demo data 
esMX_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
33512-5,Color,Tipo,Punto temporal,XXX,Nominal,,,,Color: XXX : Punto temporal: Tipo: Nominal:,,
24355-0,Panel macroscópico de análisis de orina,-,Punto temporal,Orina,-,,,,Panel macroscópico de análisis de orina: Orina : Punto temporal: -: -:,,
10003-2,Duración de la onda R. derivación III,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación III:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10006-5,Duración de la onda R. derivación V3,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V3:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10007-3,Duración de la onda R. derivación V4,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R. derivación V4:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
10024-8,Duración de la onda R 'plomo AVR,Tiempo,Punto temporal,Corazón,Cuantitativo,EKG,,,Duración de la onda R 'plomo AVR:Corazón :Punto temporal:Tiempo:Cuantitativo:EKG,,
1004-1,"Test de antiglobulina directo, reactivo específico del complemento",Presencia o umbral,Punto temporal,Eritrocitos,Ordinal,,,,"Test de antiglobulina directo, reactivo específico del complemento: Eritrocitos : Punto temporal: Presencia o umbral: Ordinal:",,
10060-2,Amplitud de onda S. Conduzca AVR,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. Conduzca AVR:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10063-6,Amplitud de onda S. derivación III,Elpot,Punto temporal,Corazón,Cuantitativo,EKG,,,Amplitud de onda S. derivación III:Corazón :Punto temporal:Elpot:Cuantitativo:EKG,,
10280-6,Número de modelo del proveedor,Tipo,Punto temporal,Tubo de cobre,Nominal,,,,Número de modelo del proveedor:Tubo de cobre :Punto temporal:Tipo:Nominal:,,
10445-5,CD11c Ag,Presencia o umbral,Punto temporal,Tejido y frotis,Ordinal,Mancha inmune,,,CD11c Ag: Tejido y frotis : Punto temporal: Presencia o umbral: Ordinal: Mancha inmune,,
10455-4,Xilosa ^30 M después de 25 g de xilosa VO,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Xilosa : Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10539-5,glipizida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,glipizida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10547-8,Primidona + FENobarbital,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Primidona + FENobarbital: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
10550-2,Temazepam,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Temazepam: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
1099-1,K sub p super sub a Ab,Presencia o umbral,Punto temporal,Suero o Plasma,Ordinal,,,,K sub p super sub a Ab: Suero o Plasma : Punto temporal: Presencia o umbral: Ordinal:,,
10995-9,Neomicina,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Neomicina: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,,
11001-5,Pirazinamida,Concentración de masa,Punto temporal,Suero o Plasma,Cuantitativo,,,,Pirazinamida: Suero o Plasma : Punto temporal: Concentración de masa: Cuantitativo:,"""

etEE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
93488-5,Guanidinoatsetaat,SCnc,Pt,Vereplekk,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Veri,
93505-6,Heptakarboksüülporfüriin I,SRat,24 tunni,U,Qn,,CHEM,,,Aine määr Kvantitatiivne Uriin,
93729-2,Beeta-2-mikroglobuliin/kreatiniin,Suhe,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
93748-2,Fibriini monomeerid,MCnc,Pt,PPP,Qn,IA,COAG,,,Juhuslik Kvantitatiivne Trombotsüütidevaene plasma,
95073-3,Histoplasma capsulatum antigeen,MCnc,Pt,BalF,Qn,IA,MICRO,,,Juhuslik Kvantitatiivne,
95074-1,Bakterid,PrThr,Pt,BalF,Ord,Valgusmikroskoopia,MICRO,,,Järgarvuline Juhuslik,
89481-6,Gentamütsiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92255-9,Metitsilliin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
92242-7,Pürasiinamiid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96635-8,HLA-C,Tüüp,Pt,B/Tis^doonor,Nom,,HLA,,,Juhuslik Kude Veri Veri või koematerjal,
95563-3,16-alfahüdroksüdehüdroepiandrosteroon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95593-0,25-hüdroksükaltsiferool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95114-5,Insuliin^2 tundi pärast sööki,Acnc,Pt,S/P,Qn,,CHAL,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
93773-0,11-deoksükortisool,SCnc,Pt,Sal,Qn,,CHEM,,,Aine kontsentratsioon Juhuslik Kvantitatiivne Sülg,
93838-1,Histoplasma capsulatum antikehad.IgM,Acnc,Pt,CSF,Qn,IA,MICRO,,,Immuunglobuliin M Juhuslik Kvantitatiivne Liikvor,
94255-7,Kaltsium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94256-5,Magneesium,MCnc,Pt,Sw,Qn,,CHEM,,,Higi Juhuslik Kvantitatiivne,
94270-6,Ubikinoon 10,SCnt,Pt,WBC,Qn,,CHEM,,,Ainehulga sisaldus Juhuslik Kvantitatiivne Leukotsüüdid,
95543-5,Aspergillus terreus antikehad.IgG,PrThr,Pt,S,Ord,,ALLERGY,,,Immuunglobuliin G Järgarvuline Juhuslik Seerum,
95527-8,Tsütomegaloviirus antikehad.IgG,PrThr,Pt,Sal,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Sülg,
95688-8,Dengue viiruse 1.+ 2.+ 3.+ 4. tüüp antikehad.IgM,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin M Järgarvuline Juhuslik Täpsustamata materjal,
93771-4,Kalprotektiin,MCnc,Pt,SynF,Qn,,CHEM,,,Juhuslik Kvantitatiivne Liigesevedelik sünoviaalvedelik,
95574-0,17-alfahüdroksüpregnanoloon,MRat,24 tunni,U,Qn,,CHEM,,,Kvantitatiivne Uriin,
95594-8,Kaltsidiool,MCnc,Pt,cB,Qn,,CHEM,,,Juhuslik Kapillaarne veri Kvantitatiivne,
95675-5,Kollapalaviku viirus antikehad.IgG,PrThr,Pt,XXX,Ord,IA,MICRO,,,Immuunglobuliin G Järgarvuline Juhuslik Täpsustamata materjal,
95719-1,Flaviviirus antikehad,PrThr,Pt,S/P,Ord,,MICRO,,,Järgarvuline Juhuslik Plasma Seerum Seerum või plasma,
95800-9,Immuunglobuliini vabad kerged ahelad.paneel,-,-,U,-,,PANEL.CHEM,,,Uriin,
95966-8,Aspergillus glaucus antikehad.IgE,Acnc,Pt,S,Qn,,ALLERGY,,,Immuunglobuliin E Juhuslik Kvantitatiivne Seerum,
96043-5,Uratsüül,MCnc,Pt,S/P,Qn,,CHEM,,,Juhuslik Kvantitatiivne Plasma Seerum Seerum või plasma,
96108-6,Klofasimiin,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,
96111-0,Linetsoliid,Susc,Pt,Is,Ord,Genotüpiseerimine,ABXBACT,,,Isolaat Järgarvuline Juhuslik Tundlikkus,"""

frBE_data = """LOINC_NUM,COMPONENT,PROPERTY,TIME_ASPCT,SYSTEM,SCALE_TYP,METHOD_TYP,CLASS,SHORTNAME,LONG_COMMON_NAME,RELATEDNAMES2,LinguisticVariantDisplayName
103631-8,Natalizumab,Concentration de masse,Temps ponctuel,Sérum,Ordinal,IA,Médicaments et produits toxiques,,,,
106016-9,Bactéries,Présence ou identité,Temps ponctuel,Pénis,Nominal,Culture,Microbiologie,,,Verge,
106033-4,Bactéries,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture anaérobique,Microbiologie,,,,
106034-2,Champignon,Présence ou identité,Temps ponctuel,Pus,Nominal,Culture,Microbiologie,,,,
103648-2,Hormone folliculo-stimulante^4 h post dose hormone de libération des gonadotrophines,Concentration arbitraire,Temps ponctuel,Sérum/Plasma,Quantitatif,,Tests de provocation,,,"4 h post dose GNRH FSH Gn-RF, Gonadotrophines-releasing factor",
103685-4,Citalopram,Concentration de masse,Temps ponctuel,Urine,Quantitatif,LC/MS/MS,Médicaments et produits toxiques,,,,
103806-6,Note,Observation,Temps ponctuel,Contact téléphonique,Document,Oncologie,DOC.CLINRPT,,,,
103830-6,Gabapentine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103834-8,Norbuprenorphine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103839-7,Phentermine,PrThr,Temps ponctuel,Méconium,Ordinal,,Médicaments et produits toxiques,,,,
103958-5,Ofloxacine,Susceptibilité,Temps ponctuel,Isolat,Ordinal,Génotypage,Sensibilité aux antibiotiques,,,,
104133-4,Éthanol,PrThr,Temps ponctuel,Gaz expiré,Ordinal,,Médicaments et produits toxiques,,,,
104181-3,Toxine du clostridium tetani,PrThr,Temps ponctuel,Sérum/Plasma,Ordinal,Test biologique sur souris,Microbiologie,,,,
104183-9,Adénovirus ADN,PrThr,Temps ponctuel,Spécimen conjonctival,Ordinal,Sonde avec amplification de la cible,Microbiologie,,,,
104196-1,Créatine/Créatinine,Ratio de substance,Temps ponctuel,Sang sur papier filtre,Quantitatif,,Chimie,,,,
104234-0,Atomoxétine,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104237-3,Zopiclone,Concentration de masse,Temps ponctuel,Urine,Quantitatif,Confirmé,Médicaments et produits toxiques,,,,
104419-7,Legionella sp Ac^1er échantillon,Titre,Temps ponctuel,Sérum,Ordinal,IA,Microbiologie,,,Anticorps Echantillon.1,
104457-7,Virus varicelle-zona Anticorps.IgA,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,
104459-3,Virus varicelle-zona Anticorps.IgG,PrThr,Temps ponctuel,LCR,Ordinal,IA,Microbiologie,,,Anticorps VZV,"""


In [168]:
### FIELD MAPPING DATASET - LOINCs ###

# This dictionary maps LOINC CSV fields to their corresponding OCL Concept fields.
# Mappings are based on the provided PDF and the LOINC FHIR example.
loinc_to_ocl_mapping = {    
    # General Format: '[Loinc Field]':'[OCL field]'

    #Field Mappings
    'LOINC_NUM': ['id','extras.Code_in_Source'],
    'LONG_COMMON_NAME': ['names.Fully-Specified.en[1]','extras.LONG_COMMON_NAME'],
    'DisplayName': ['names.Display.en[1]','extras.DisplayName'],
    'SHORTNAME': ['names.Short.en[1]','extras.SHORTNAME'],
    'CONSUMER_NAME': ['names.Consumer.en[1]','extras.CONSUMER_NAME'],
    'SCALE_TYP': ['datatype','extras.SCALE_TYP'],
    'STATUS': ['retired','extras.STATUS'], # Retired translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
    'DefinitionDescription': ['description','extras.DEFINITION_DESCRIPTION'],

    # Mappings derived from LOINC FHIR example JSON
    # These will be stored as `extras` to align with the FHIR properties.
    'COMPONENT': 'extras.COMPONENT',
    'PROPERTY': 'extras.PROPERTY',
    'TIME_ASPCT': 'extras.TIME_ASPCT',
    'SYSTEM': 'extras.SYSTEM',
    'METHOD_TYP': 'extras.METHOD_TYP',
    'VersionFirstReleased':'extras.VersionFirstReleased',
    'VersionLastChanged':'extras.VersionLastChanged',
    'ORDER_OBS':'extras.ORDER_OBS',
    'HL7_FIELD_SUBFIELD_ID':'extras.HL7_FIELD_SUBFIELD_ID',
    'EXTERNAL_COPYRIGHT_NOTICE':'extras.EXTERNAL_COPYRIGHT_NOTICE',
    'SURVEY_QUEST_TEXT':'extras.SURVEY_QUEST_TEXT',
    'SURVEY_QUEST_SRC':'extras.SURVEY_QUEST_SRC',
    'UNITSREQUIRED':'extras.UNITSREQUIRED',
    'RELATEDNAMES2':'extras.RELATEDNAMES2',
    'EXTERNAL_COPYRIGHT_LINK': 'extras.EXTERNAL_COPYRIGHT_LINK',
    'ValidHL7AttachmentRequest': 'extras.ValidHL7AttachmentRequest',
    'CHNG_TYPE': 'extras.CHNG_TYPE',
    'STATUS_TEXT': 'extras.STATUS_TEXT',
    'STATUS_REASON': 'extras.STATUS_REASON',
    'PanelType': 'extras.PanelType',
    'CHANGE_REASON_PUBLIC': 'extras.CHANGE_REASON_PUBLIC',
    'COMMON_TEST_RANK': 'extras.COMMON_TEST_RANK',
    'AskAtOrderEntry': 'extras.AskAtOrderEntry',
    'AssociatedObservations': 'extras.AssociatedObservations',
    'EXAMPLE_UNITS': 'extras.EXAMPLE_UNITS',
    'EXMPL_ANSWERS': 'extras.EXMPL_ANSWERS',
    'EXAMPLE_UCUM_UNITS': 'extras.EXAMPLE_UCUM_UNITS',
    'HL7_ATTACHMENT_STRUCTURE': 'extras.HL7_ATTACHMENT_STRUCTURE',
    'COMMON_ORDER_RANK': 'extras.COMMON_ORDER_RANK',
    'FORMULA': 'extras.FORMULA',
    'CLASS': 'extras.CLASS',
    'CLASSTYPE': 'extras.CLASSTYPE'

}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc = {
    'type': 'Concept',
    'concept_class': 'LOINC',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'extras.Code_Type': 'LOINC'
}

In [169]:
### FIELD MAPPING DATASET - LOINC Parts ###

loinc_part_to_ocl_mapping ={
    "PartNumber": ["id", "extras.Code_in_Source"],
    "PartTypeName": "extras.PartTypeName",
    "PartName": ["names.Fully-Specified.en[1]", "extras.LONG_COMMON_NAME"],
    "PartDisplayName": ["names.Display.en[1]", "extras.PartDisplayName"],
    'Status': ['retired','extras.STATUS'] # 'retired' translation: ACTIVE/TRIAL/DISCOURAGED → false, DEPRECATED → true
}

# This dictionary contains the fixed values to be used in the transformation.
fixed_values_loinc_parts = {
    'type': 'Concept',
    'concept_class': 'LOINC Part',
    'datatype': 'N/A',
    'source': 'LOINC',
    'owner_type': 'Organization',
    'owner': 'Regenstrief',
    'extras.Code_Type': 'LOINC Part'
}

In [170]:
### FIELD MAPPING DATASET - LOINC Answer Lists and Answers ###

# Answer Lists
answer_list_to_ocl_mapping = {
    "AnswerListId": ["id", "extras.Code_in_Source"],
    "AnswerListName": ["names.Fully-Specified.en[1]", "extras.AnswerListName"],
    "AnswerListOID": "extras.AnswerListOID",
    "ExtDefinedYN": "extras.ExtDefinedYN",
    "ExtDefinedAnswerListCodeSystem": "extras.ExtDefinedAnswerListCodeSystem",
    "ExtDefinedAnswerListLink": "extras.ExtDefinedAnswerListLink"
}

fixed_values_answer_list = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "Answer List",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "extras.Code_Type": "LOINC Answer List"
}

# Answers
answer_to_ocl_mapping = {
    "AnswerStringId": ["id", "extras.Code_in_Source"],
    "DisplayText": ["names.Display.en[1]", "extras.DisplayText"],
    "LocalAnswerCode": "extras.LocalAnswerCode",
    "LocalAnswerCodeSystem": "extras.LocalAnswerCodeSystem",
    "SequenceNumber": "extras.SequenceNumber",
    "ExtCodeId": "extras.ExtCodeId",
    "ExtCodeDisplayName": ["names.Fully-Specified.en[1]", "extras.ExtCodeDisplayName"],
    "ExtCodeSystem": "extras.ExtCodeSystem",
    "ExtCodeSystemVersion": "extras.ExtCodeSystemVersion",
    "ExtCodeSystemCopyrightNotice": "extras.ExtCodeSystemCopyrightNotice",
    "SubsequentTextPrompt": "extras.SubsequentTextPrompt",
    "Description": "extras.Description",
    "Score": "extras.Score"
}

fixed_values_answer = {
    'type': 'Concept',
    'retired': False,
    "concept_class": "LOINC Answer",
    "datatype": "N/A",
    "source": "LOINC",
    "owner_type": "Organization",
    "owner": "Regenstrief",
    "extras.Code_Type": "LOINC Answer"
}

In [171]:
#Transformation rules - specify fields to transform and their target values

transformation_rules =  [
    {
      "field": ["STATUS", "Status"],
      "transformations": {
        "DEPRECATED": True,
        "ACTIVE": False,
        "TRIAL": False,
        "DISCOURAGED": False
      },
      "target_field": "retired"
    }
  ]

In [172]:
### Mapping Configurations ###

# Define the base URL for OCL concepts
CONCEPT_URL_PREFIX = "/orgs/Regenstrief/sources/LOINC/concepts/{}/"

# Define the overall configuration dictionary
MAPPING_CONFIGS = {
    "Panel-to-Test": {
        "source_files": ["PanelsAndForms.csv"],
        "map_type": "has element",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "has element"},
            {"ocl_field": "from_concept_url", "source_field": "ParentLoinc", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "Loinc", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["Sequence"]', "source_field": "SequenceInPanel"},
            {"ocl_field": 'extras["Required"]', "source_field": "Required"},
            {
                "ocl_field": 'extras["Cardinality"]',
                "source_field_min": "CardinalityMin",
                "source_field_max": "CardinalityMax",
                "rule": lambda min_val, max_val: f"{min_val}..{max_val}",
            },
            {"ocl_field": 'extras["Answer List Override"]', "source_field": "AnswerListIdOverride"},
            {"ocl_field": 'extras["Answer List Type Override"]', "source_field": "AnswerListTypeOverride"},
        ],
    },
    "Question-to-Answer": {
        "source_files": ["LoincAnswerListLink.csv", "AnswerList.csv"],
        "map_type": "has answer",
        "join_condition": {"left_on": "LoincAnswerListLink.AnswerListId", "right_on": "AnswerList.AnswerListId"},
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "has answer"},
            {"ocl_field": "from_concept_url", "source_field": "LoincNumber", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AnswerStringId", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["Answer List ID"]', "source_field": "LoincAnswerListLink.AnswerListId"},
            {"ocl_field": 'extras["Answer List Type"]', "source_field": "LoincAnswerListLink.AnswerListLink Type"},
            {"ocl_field": 'extras["Sequence"]', "source_field": "AnswerList.SequenceNumber"},
            {"ocl_field": 'extras["Score"]', "source_field": "AnswerList.Score"},
            {"ocl_field": 'extras["Local Answer Code"]', "source_field": "AnswerList.LocalAnswerCode"},
        ],
    },
    "Ask at Order Entry": {
        "source_files": ["Loinc.csv"],
        "map_type": "Ask At Order Entry",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Ask At Order Entry"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AskAtOrderEntry", "rule": lambda value: CONCEPT_URL_PREFIX.format(value) if pd.notna(value) else None},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
        ],
    },
    "Code Evolution": {
        "source_files": ["MapTo.csv"],
        "map_type": "Map To",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Map To"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "MAP_TO", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
            {"ocl_field": 'extras["COMMENT"]', "source_field": "COMMENT"},
        ],
    },
    "Associated Observations": {
        "source_files": ["Loinc.csv"],
        "map_type": "Associated Observations",
        "field_mappings": [
            {"ocl_field": "type", "source_value": "Mapping"},
            {"ocl_field": "map_type", "source_value": "Associated Observations"},
            {"ocl_field": "from_concept_url", "source_field": "LOINC_NUM", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "to_concept_url", "source_field": "AssociatedObservations", "rule": CONCEPT_URL_PREFIX.format},
            {"ocl_field": "source", "source_value": "LOINC"},
            {"ocl_field": "owner_type", "source_value": "Organization"},
            {"ocl_field": "owner", "source_value": "Regenstrief"},
        ],
    },
}

In [173]:
###  Hierarchy Configurations  ###

hierarchy_mapping = {
    
    #Field Mappings
    # PATH_TO_ROOT,SEQUENCE,IMMEDIATE_PARENT,CODE,CODE_TEXT

    'PATH_TO_ROOT': 'extras.PATH_TO_ROOT',
    'SEQUENCE': 'extras.HIERARCHY_SEQUENCE',
    'IMMEDIATE_PARENT': 'parent_concept', # This will become the parent concept URL
    'CODE': 'match-id' # This will be used to match with the LOINC or Part concept ID
    # 'CODE_TEXT' will be ignored
}

## Phase 1: Data Loading and Validation

In [174]:
# Data Loading and Validation - Concepts

def load_data():
    """Loads data based on the `mode` variable and validates fields against the mapping."""
    if mode == 2:
        loinc_df = demo_loinc_df
        part_link_df = demo_part_link_df
        answer_list_df = demo_answer_list_df
    elif mode == 1:
        # In test mode, we load a small subset of the real data
        try:
            loinc_df = pd.read_csv(loinc_csv_path, nrows=15, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, nrows=15, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, nrows=15, low_memory=False)
    elif mode == 0:
        # In full run mode, we load the entire CSV files
        try:
            loinc_df = pd.read_csv(loinc_csv_path, low_memory=False)
        except FileNotFoundError:
            loinc_df = pd.DataFrame()
            print(f"⚠️ Warning: The file '{loinc_csv_path}' was not found. LOINC validation will be skipped.")
            
        part_link_df = pd.read_csv(part_link_csv_path, low_memory=False)
        answer_list_df = pd.read_csv(answer_list_csv_path, low_memory=False)
    else:
        print("Invalid mode selected. Please use 0, 1, or 2.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # ... (rest of the load_data function remains the same) ...
    # The validation logic below is not changed.

    # A helper function to create a DataFrame from the raw CSV data
    def load_csv_data(csv_data):
        return pd.read_csv(StringIO(csv_data))

    # This part needs to be changed. The code below should dynamically
    # load files from the linguistic_variants_path directory for modes 0 and 1.
    if mode in [0, 1]:
        print("Loading linguistic variant files...")
        linguistic_variant_dfs = []
        for filename in os.listdir(linguistic_variants_path):
            if filename.endswith('.csv'):
                filepath = os.path.join(linguistic_variants_path, filename)
                df = pd.read_csv(filepath, low_memory=False)
                linguistic_variant_dfs.append(df)
    else: # mode == 2
        # Use hardcoded data for demo mode
        esMX_df = load_csv_data(esMX_data)
        etEE_df = load_csv_data(etEE_data)
        frBE_df = load_csv_data(frBE_data)
        linguistic_variant_dfs = [esMX_df, etEE_df, frBE_df]
    
    print(f"Loaded {len(linguistic_variant_dfs)} linguistic variant files.")
    
    return loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs

# Call the function to load all data
loinc_df, part_link_df, answer_list_df, linguistic_variant_dfs = load_data()

def load_and_preprocess_hierarchy_data(file_path, column_map):
    """
    Loads the ComponentHierarchyBySystem CSV, renames and selects columns
    based on the provided mapping, and handles initial data types.
    """
    try:
        df = pd.read_csv(file_path)
        
        # Select only the columns specified in the mapping.
        # The key represents the new column name, and the value is the old column name.
        df = df[list(column_map.keys())]
        
        # Rename the selected columns using the mapping.
        df = df.rename(columns=column_map)
        
        return df
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None

# Load the data into a DataFrame.
hierarchy_df = load_and_preprocess_hierarchy_data(component_hierarchy_file_path, hierarchy_mapping)

if hierarchy_df is not None:
    print("Successfully loaded and renamed hierarchy data with the new mapping:")
    # print(hierarchy_df.head())

Loading linguistic variant files...
Loaded 3 linguistic variant files.
Successfully loaded and renamed hierarchy data with the new mapping:


In [175]:
# Adds linguistic variant file names, dynamically generated from the directory.

linguistic_variant_files = [os.path.join(linguistic_variants_path, f) for f in os.listdir(linguistic_variants_path) if f.endswith('.csv')]

# Function to create the Fully Specified Name (FSN)
def create_fsn(row):
    parts = [row['COMPONENT'], row['PROPERTY'], row['TIME_ASPCT'], row['SYSTEM'], row['SCALE_TYP'], row['METHOD_TYP'], row['CLASS']]
    return ':'.join([str(part) for part in parts if pd.notna(part) and str(part).strip() != ''])

# Function to process a single linguistic variant file and add the locale code
def process_linguistic_file(file_name):
    locale_code = os.path.basename(file_name)[:2]
    df = pd.read_csv(file_name, low_memory=False)
    processed_rows = []
    
    for index, row in df.iterrows():
        fsn = create_fsn(row)
        if fsn:
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': fsn, 'NameType': 'Fully-Specified'
            })
        if pd.notna(row['SHORTNAME']) and row['SHORTNAME'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['SHORTNAME'], 'NameType': 'Short'
            })
        if pd.notna(row['LONG_COMMON_NAME']) and row['LONG_COMMON_NAME'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['LONG_COMMON_NAME'], 'NameType': 'Display'
            })
        if pd.notna(row['RELATEDNAMES2']) and row['RELATEDNAMES2'].strip() != '':
            processed_rows.append({
                'LOINC_NUM': row['LOINC_NUM'], 'LocaleCode': locale_code,
                'LinguisticVariantName': row['RELATEDNAMES2'], 'NameType': 'None'
            })
            
    return pd.DataFrame(processed_rows)

# Process all linguistic variant files and concatenate the results
all_processed_variants = pd.concat(
    [process_linguistic_file(file) for file in linguistic_variant_files], 
    ignore_index=True
)


# Create a copy to avoid modifying the original dataframe
temp_df = all_processed_variants.copy()

# Sort the data to ensure the numeric counter is assigned consistently.
temp_df.sort_values(by=['LOINC_NUM', 'NameType', 'LocaleCode'], inplace=True)

# Generate a numeric counter for each unique combination of LOINC_NUM, NameType, and LocaleCode.
temp_df['counter'] = temp_df.groupby(['LOINC_NUM', 'NameType', 'LocaleCode']).cumcount() + 1

# Create a new column that will be used for the pivot operation.
# The format will be 'names.[NameType].[LocaleCode][counter]'.
temp_df['pivot_column'] = 'names.' + temp_df['NameType'].astype(str) + '.' + temp_df['LocaleCode'].astype(str) + '[' + temp_df['counter'].astype(str) + ']'

# Pivot the DataFrame using the new 'pivot_column' and 'LinguisticVariantName' values.
pivoted_variants = temp_df.pivot(index='LOINC_NUM', columns='pivot_column', values='LinguisticVariantName')

# Flatten the MultiIndex columns and reset the index.
pivoted_variants.columns = pivoted_variants.columns.get_level_values(0)
pivoted_variants.reset_index(inplace=True)

# Merge the pivoted variants with the main LOINC DataFrame.
merged_loinc_df = pd.merge(loinc_df, pivoted_variants, on='LOINC_NUM', how='left')

# To see the pivoted result, you can display the head of the new dataframe
# print("\nPivoted Variants DataFrame head with new linguistic variant columns:")
# print(pivoted_variants.head())

## Phase 2: Concept Creation

In this phase, we will transform LOINC terms, parts, answer lists, and linguistic variants into OCL Concept objects.

Phase 2 will receive the following from Phase 1:

    merged_loinc_df: A pandas DataFrame containing data from the Loinc.csv file, enriched with linguistic variants from all the linguistic variant CSV files. This DataFrame includes a wide range of LOINC fields and new columns for each linguistic variant and its locale.

    part_link_df: A pandas DataFrame loaded from the Part.csv file, containing information about LOINC parts, such as PartNumber, PartName, and PartTypeName.

    answer_list_df: A pandas DataFrame from the AnswerList.csv file, which includes details about LOINC answer lists and their corresponding answers.

### Expected Outputs of Phase 2

The output of Phase 2 will be a collection of OCL Concept objects, represented as a list of dictionaries. These objects will be generated from the dataframes provided by Phase 1 and will be structured to match the OCL format for Phase 5 ouputting. Specifically, this phase will produce the following dataframes:

    LOINC Concept Objects: Dictionaries that represent each LOINC code as an OCL concept, including its primary names, linguistic variants, and other attributes mapped to the extras field.

    LOINC Part Concept Objects: Dictionaries representing each unique LOINC part, with its PartNumber, name, and type properly mapped.

    LOINC Answer List Concept Objects: Dictionaries for each answer list, containing its ID and name.

    LOINC Answer Concept Objects: Dictionaries for each individual answer, linked to its parent answer list, including the display text and sequence number.

### Actions to be Taken

To produce the expected outputs, the code in Phase 2 will perform the following actions:

    Iterate and Map LOINC Data: It will loop through each row of the merged_loinc_df DataFrame. For each row, it will apply the loinc_to_ocl_mapping and fixed_values_loinc dictionaries to rename the LOINC fields. It will also apply the transformation_rules e.g. to set the retired status based on the STATUS field, along with other defined rules as needed. The new linguistic variant columns will be merged with the LOINC data to provide additional fields based on language.

    Create LOINC Part Concepts: The code will process the part_link_df DataFrame to create a distinct OCL Concept object for each unique PartNumber. It will use the loinc_part_to_ocl_mapping and fixed_values_loinc_parts to populate the OCL-specific fields.

    Generate Answer List and Answer Concepts: It will iterate through the answer_list_df DataFrame to create OCL Concept datasets, one each for LOINC Answers and Answer Lists. It will first create an object for each unique answer list (using answer_list_to_ocl_mapping and fixed_values_answer_list), and then create a separate OCL Concept object for each individual answer within those lists (using answer_to_ocl_mapping and fixed_values_answer). Duplicate rows will be deduplicated.

    Consolidate Concepts: All generated OCL Concept objects (for LOINCs, LOINC Parts, Answer Lists, and Answers) will be collected into a single, comprehensive list of dictionaries. This consolidated list will be the primary output for use in subsequent phases.

In [176]:
## Functions for creating OCL concepts for all LOINC types

def check_for_unmapped_fields(df, mapping, name_of_df, ignored_cols=None):
    """
    Checks for unmapped columns in a DataFrame and prints a warning message.
    """
    if ignored_cols is None:
        ignored_cols = []
    
    mapped_source_fields = set(mapping.keys())
    
    # Get all columns from the DataFrame that are not in the ignored list
    df_columns = set(df.columns) - set(ignored_cols)
    
    unmapped_fields = df_columns - mapped_source_fields
    
    if unmapped_fields:
        print(f"⚠️ Warning: Unmapped fields found in '{name_of_df}': {', '.join(sorted(list(unmapped_fields)))}")
        
def create_ocl_concept_from_row(row, mapping, fixed_values, transformation_rules=None):
    """
    Creates a single OCL Concept dictionary from a pandas DataFrame row,
    applying the specified mapping, fixed values, and transformation rules.
    This version keeps all attributes as top-level fields.
    """
    concept = fixed_values.copy()
    
    # Process column mappings
    for source_field, target_fields in mapping.items():
        if source_field in row and pd.notna(row[source_field]):
            if isinstance(target_fields, list):
                for target_field in target_fields:
                    concept[target_field] = row[source_field]
            else:
                concept[target_fields] = row[source_field]
    
    # Process transformation rules
    if transformation_rules:
        for rule in transformation_rules:
            fields_to_check = rule['field'] if isinstance(rule['field'], list) else [rule['field']]
            
            for field in fields_to_check:
                # Check for the existence of the field and handle both existing and missing values
                if field in row:
                    source_value = row[field]
                    
                    # Special handling for NaN, which cannot be a dictionary key
                    # Check if the value is NaN and if the rule has a transformation for NaN
                    if pd.isna(source_value):
                        if np.nan in rule['transformations']:
                            target_value = rule['transformations'][np.nan]
                            concept[rule['target_field']] = target_value
                            break # Stop after finding the first matching field and transforming
                    elif source_value in rule['transformations']:
                        target_value = rule['transformations'][source_value]
                        concept[rule['target_field']] = target_value
                        break # Stop after finding the first matching field and transforming
    return concept

def process_loinc_concepts(df):
    """
    Creates OCL Concept objects for all LOINC codes, including linguistic variants,
    with all attributes as top-level fields.
    """
    loinc_concepts = []
    
    linguistic_variant_cols = [col for col in df.columns if col.startswith('names.')]
    
    # Check for unmapped fields in the DataFrame
    ignored_cols = linguistic_variant_cols + ['PartNumber', 'ANSWERLISTID']
    check_for_unmapped_fields(df, loinc_to_ocl_mapping, 'merged_loinc_df', ignored_cols=ignored_cols)
    
    for _, row in df.iterrows():
        base_loinc_concept = create_ocl_concept_from_row(row, loinc_to_ocl_mapping, fixed_values_loinc, transformation_rules)

        # Handle dynamic linguistic variant names
        for col in linguistic_variant_cols:
            if pd.notna(row[col]):
                base_loinc_concept[col] = row[col]
        
        loinc_concepts.append(base_loinc_concept)
        
    return loinc_concepts

def process_loinc_parts(df):
    """
    Creates OCL Concept objects for unique LOINC Parts with all attributes
    as top-level fields.
    """
    loinc_part_concepts = []
    processed_part_numbers = set()
    
    # Check for unmapped fields
    check_for_unmapped_fields(df, loinc_part_to_ocl_mapping, 'part_link_df')

    for _, row in df.iterrows():
        part_number = row['PartNumber']
        if part_number not in processed_part_numbers:
            concept = create_ocl_concept_from_row(
                row, loinc_part_to_ocl_mapping, fixed_values_loinc_parts, transformation_rules
            )
            loinc_part_concepts.append(concept)
            processed_part_numbers.add(part_number)
            
    return loinc_part_concepts

def process_answer_lists_and_answers(df):
    """
    Creates OCL Concept objects for both LOINC Answer Lists and individual Answers,
    with all attributes as top-level fields.
    Deduplicates concepts to ensure uniqueness.
    """
    answer_list_concepts = []
    answer_concepts = []
    processed_answer_list_ids = set()
    processed_answer_ids = set()
    
    # Check for unmapped fields
    check_for_unmapped_fields(df, {**answer_list_to_ocl_mapping, **answer_to_ocl_mapping}, 'answer_list_df')

    for _, row in df.iterrows():
        # Create Answer List Concept
        answer_list_id = row['AnswerListId']
        if answer_list_id not in processed_answer_list_ids:
            answer_list_concept = create_ocl_concept_from_row(
                row, answer_list_to_ocl_mapping, fixed_values_answer_list, transformation_rules
            )
            answer_list_concepts.append(answer_list_concept)
            processed_answer_list_ids.add(answer_list_id)

        # Create Answer Concept
        answer_string_id = row['AnswerStringId']
        if answer_string_id not in processed_answer_ids:
            answer_concept = create_ocl_concept_from_row(
                row, answer_to_ocl_mapping, fixed_values_answer, transformation_rules
            )
            answer_concept['ParentAnswerListId'] = answer_list_id
            
            answer_concepts.append(answer_concept)
            processed_answer_ids.add(answer_string_id)
            
    return answer_list_concepts, answer_concepts

In [177]:
def phase_2_main(merged_loinc_df, part_link_df, answer_list_df):
    """
    Main function to execute all Phase 2 concept creation tasks and
    consolidate the outputs into a single list.
    """
    # Create LOINC Concepts
    print("Creating LOINC concepts...")
    loinc_concepts = process_loinc_concepts(merged_loinc_df)
    print(f"Created {len(loinc_concepts)} LOINC concepts from `merged_loinc_df`.")

    # Create LOINC Part Concepts
    print("Creating LOINC Part concepts...")
    loinc_part_concepts = process_loinc_parts(part_link_df)
    print(f"Created {len(loinc_part_concepts)} LOINC Part concepts from `part_link_df`.")

    # Create Answer List and Answer Concepts
    print("Creating Answer List and Answer concepts...")
    answer_list_concepts, answer_concepts = process_answer_lists_and_answers(answer_list_df)
    print(f"Created {len(answer_list_concepts)} Answer List concepts and {len(answer_concepts)} Answer concepts from `answer_list_df`.")
    
    # Consolidate all concepts into a single list
    all_concepts = loinc_concepts + loinc_part_concepts + answer_list_concepts + answer_concepts
    
    return all_concepts

# Execute the main function of Phase 2
all_ocl_concepts_list = phase_2_main(merged_loinc_df, part_link_df, answer_list_df)

# Convert the list of concepts into a pandas DataFrame.
all_ocl_concepts_df = pd.DataFrame(all_ocl_concepts_list)

# Now, sort the DataFrame's columns alphabetically.
all_ocl_concepts_df_sorted = all_ocl_concepts_df.sort_index(axis=1)

# Print the total number of concepts created.
print(f"\nTotal OCL Concepts created in Phase 2: {len(all_ocl_concepts_df_sorted)}")

# The sorted DataFrame is now stored in `all_ocl_concepts_df_sorted`.

Creating LOINC concepts...
Created 154 LOINC concepts from `merged_loinc_df`.
Creating LOINC Part concepts...
Created 106 LOINC Part concepts from `part_link_df`.
Creating Answer List and Answer concepts...
Created 46 Answer List concepts and 205 Answer concepts from `answer_list_df`.

Total OCL Concepts created in Phase 2: 511


## Phase 3: Mapping Creation

This phase is for creating OCL Mapping objects based on relational files like `PanelsAndForms.csv` and `MapTo.csv`.

In [178]:
# --- DATA LOADING AND PREPARATION ---


try:
    df_panels_and_forms = pd.read_csv(panels_and_forms_csv_path, low_memory=False)
    df_loinc_answer_list_link = pd.read_csv(loinc_answer_list_link_csv_path, low_memory=False)
    df_answer_list = pd.read_csv(answer_list_csv_path, low_memory=False)
    df_loinc = pd.read_csv(loinc_csv_path, low_memory=False)
    df_map_to = pd.read_csv(map_to_csv_path, low_memory=False)
    print("All DataFrames loaded successfully.")
    
except NameError:
    # Fallback to hardcoded file names if path variables are not defined.
    print("Warning: Path variables not found. Attempting to load from hardcoded filenames.")
    try:
        df_panels_and_forms = pd.read_csv("PanelsAndForms.csv")
        df_loinc_answer_list_link = pd.read_csv("LoincAnswerListLink.csv")
        df_answer_list = pd.read_csv("AnswerList.csv")
        df_loinc = pd.read_csv("Loinc.csv")
        df_map_to = pd.read_csv("MapTo.csv")
        print("DataFrames loaded from hardcoded filenames.")
    except FileNotFoundError as e:
        print(f"Error: One or more files not found: {e}")
        # Initialize empty DataFrames to prevent further errors
        df_panels_and_forms = pd.DataFrame()
        df_loinc_answer_list_link = pd.DataFrame()
        df_answer_list = pd.DataFrame()
        df_loinc = pd.DataFrame()
        df_map_to = pd.DataFrame()

except FileNotFoundError as e:
    print(f"Error: One or more files not found: {e}")
    # Initialize empty DataFrames to prevent further errors
    df_panels_and_forms = pd.DataFrame()
    df_loinc_answer_list_link = pd.DataFrame()
    df_answer_list = pd.DataFrame()
    df_loinc = pd.DataFrame()
    df_map_to = pd.DataFrame()

All DataFrames loaded successfully.


In [179]:
# This cell contains the core function to process the mappings.

# This cell contains the core function to process the mappings.
def process_mappings(df, config):
    """
    Generates OCL mapping objects from a DataFrame using a configuration dictionary.
    """
    mapping_objects = []
    
    # Process each row of the input DataFrame
    for _, row in df.iterrows():
        ocl_object = {}
        
        # Populate fixed and mapped fields
        for field_map in config["field_mappings"]:
            ocl_field = field_map["ocl_field"]
            
            # Handle fixed values
            if "source_value" in field_map:
                value = field_map["source_value"]
            # Handle direct field mappings with an optional rule
            elif "source_field" in field_map:
                source_field = field_map["source_field"]
                value = row.get(source_field)
                if "rule" in field_map:
                    if pd.notna(value):
                        value = field_map["rule"](value)
                    else:
                        # NEW: Assign None if the source value is not a number and no rule is applied
                        value = None
            # Handle combined field mappings (e.g., for Cardinality)
            elif "source_field_min" in field_map and "source_field_max" in field_map:
                min_val = row.get(field_map["source_field_min"])
                max_val = row.get(field_map["source_field_max"])
                if pd.notna(min_val) and pd.notna(max_val):
                    value = field_map["rule"](min_val, max_val)
                else:
                    value = None
            else:
                continue

            # Populate the OCL object, handling the 'extras' dictionary
            if ocl_field.startswith('extras'):
                if "extras" not in ocl_object:
                    ocl_object["extras"] = {}
                key = ocl_field.split('"')[1]
                ocl_object["extras"][key] = value
            else:
                ocl_object[ocl_field] = value
        
        # Only append valid OCL objects if both `from_concept_url` and `to_concept_url` are present.
        if ocl_object.get("from_concept_url") and ocl_object.get("to_concept_url"):
            mapping_objects.append(ocl_object)
            
    return mapping_objects

# --- MAPPING EXECUTION ---

# Assuming your DataFrames are already loaded (e.g., df_loinc, df_panels_and_forms, etc.)

# Initialize mapping lists to empty
panel_to_test_mappings = []
Youtube_mappings = []
order_entry_mappings = []
code_evolution_mappings = []
associated_observations_mappings = []

# Process the Panel-to-Test mappings.
panel_to_test_mappings = process_mappings(df_panels_and_forms, MAPPING_CONFIGS["Panel-to-Test"])
print(f"Generated {len(panel_to_test_mappings)} Panel-to-Test mappings.")
print("-" * 20)

# Process the Question-to-Answer mappings.
join_info = MAPPING_CONFIGS["Question-to-Answer"]["join_condition"]
left_col = join_info["left_on"].split('.')[-1]
right_col = join_info["right_on"].split('.')[-1]
df_joined_qa = pd.merge(df_loinc_answer_list_link, df_answer_list, left_on=left_col, right_on=right_col, suffixes=("_LoincAnswerListLink", "_AnswerList"))
Youtube_mappings = process_mappings(df_joined_qa, MAPPING_CONFIGS["Question-to-Answer"])
print(f"Generated {len(Youtube_mappings)} Question-to-Answer mappings.")
print("-" * 20)

# Process the Ask at Order Entry mappings.
order_entry_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Ask at Order Entry"])
print(f"Generated {len(order_entry_mappings)} Ask at Order Entry mappings.")
print("-" * 20)

# Process the Code Evolution mappings.
code_evolution_mappings = process_mappings(df_map_to, MAPPING_CONFIGS["Code Evolution"])
print(f"Generated {len(code_evolution_mappings)} Code Evolution mappings.")
print("-" * 20)

# Process the Associated Observations mappings.
associated_observations_mappings = process_mappings(df_loinc, MAPPING_CONFIGS["Associated Observations"])
print(f"Generated {len(associated_observations_mappings)} Associated Observations mappings.")
print("-" * 20)

# Combine all mappings into a single list for the final output
all_mappings = (
    panel_to_test_mappings +
    Youtube_mappings +
    order_entry_mappings +
    code_evolution_mappings +
    associated_observations_mappings
)
print(f"\nTotal OCL mapping objects created for Round 1: {len(all_mappings)}")

Generated 1064 Panel-to-Test mappings.
--------------------
Generated 312 Question-to-Answer mappings.
--------------------
Generated 3 Ask at Order Entry mappings.
--------------------
Generated 999 Code Evolution mappings.
--------------------
Generated 3 Associated Observations mappings.
--------------------

Total OCL mapping objects created for Round 1: 2381


## Phase 4: Hierarchy Creation

Phase 4 focuses on parsing the `ComponentHierarchyBySystem.csv` file and establishing hierarchy relationships between concepts. One single hierarchy will be created in the end, which means that every concept will have a field 'parent_concept_urls' with one or more concept URLs (i.e. the URL prefix plus the ID of the parent concept). One example: "parent_concept_urls":["/orgs/Regenstrief/sources/LOINC-3/concepts/ROOT/"]

The Hierarchy includes:
* Component-by-System Hierarchy - Connects LOINC Parts or LOINCs to a parent part
* Top-level Containers - Connects parts and terms without parents to a Container concept
* List-to-Answer Hierarchy - Links LOINC Answers to their Answer List parent(s)

This phase creates a set of Container concepts, whose parent concept is the special "ROOT" concept. ROOT is the only concept without a parent. This Phase should assign parent concept URLs, along with identifying and listing concepts (grouped by their concept_class) that do not yet have a parent.

In [180]:
### Hierarchy Analysis Functions ###

def identify_node_types(df):
    """
    Identifies LOINC Parts (branches) and LOINC terms (leaf nodes) in the hierarchy.
    LOINC Parts are the branches, and LOINC terms are the leaf nodes.
    """
    # LOINC Parts (branches) appear in the 'parent_concept' column.
    loinc_parts = df['parent_concept'].dropna().unique()
    # LOINC terms (leaf nodes) are 'match-id' values that are not 'parent_concept' values.
    loinc_terms = df[~df['match-id'].isin(loinc_parts)]['match-id'].unique()
    
    print("\nLOINC Hierarchy Breakdown:")
    print(f"Total unique codes: {df['match-id'].nunique()}")
    print(f"Number of LOINC Parts (branches): {len(loinc_parts)}")
    print(f"Number of LOINC Terms (leaf nodes): {len(loinc_terms)}")
    return loinc_parts, loinc_terms

# Execute the analysis functions if data was loaded successfully.
if hierarchy_df is not None:
    loinc_parts, loinc_terms = identify_node_types(hierarchy_df)
else:
    print("Error: The hierarchy_df is not available. Please check the data loading step.")


LOINC Hierarchy Breakdown:
Total unique codes: 493
Number of LOINC Parts (branches): 133
Number of LOINC Terms (leaf nodes): 360


In [181]:
# Rename columns in hierarchy_df based on the provided mapping before merging.
# This code assumes hierarchy_df and hierarchy_mapping are already defined.
hierarchy_df = hierarchy_df.rename(columns=hierarchy_mapping)

# Merge the 'all_ocl_concepts_df_sorted' and the renamed 'hierarchy_df'.
# This merge is a left outer join, keeping all records from the concepts dataframe
# and only adding matching records from the hierarchy dataframe.
concepts_with_hierarchy_df = pd.merge(all_ocl_concepts_df_sorted, hierarchy_df, left_on='id', right_on='match-id', how='left')

# Sort the columns of the merged dataframe for better organization.
concepts_with_hierarchy_df = concepts_with_hierarchy_df.sort_index(axis=1)

# Identify concepts with and without a parent concept based on the 'parent_concept' column.
# A concept is considered to have a parent if the 'parent_concept' column is not null.
concepts_with_parent = concepts_with_hierarchy_df[concepts_with_hierarchy_df['parent_concept'].notna()]
concepts_without_parent = concepts_with_hierarchy_df[concepts_with_hierarchy_df['parent_concept'].isna()]

# Report the findings.
total_concepts = len(concepts_with_hierarchy_df)
num_with_parent = len(concepts_with_parent)
num_without_parent = len(concepts_without_parent)

print(f"Total number of concepts across all datasets: {total_concepts}")
print(f"Number of concepts with a parent concept: {num_with_parent}")
print(f"Number of concepts without a parent concept: {num_without_parent}")

Total number of concepts across all datasets: 511
Number of concepts with a parent concept: 10
Number of concepts without a parent concept: 501


In [182]:
# Create the parent_concept_urls list.
def create_parent_urls(row):
    """
    Creates a list of parent concept URLs for a given row.
    """
    parent_urls = []
    
    # Add the parent_concept URL if the value is not null.
    if pd.notna(row.get('parent_concept')):
        parent_urls.append(CONCEPT_URL_PREFIX.format(row['parent_concept']))
    
    # Add the ParentAnswerListId URL if the value is not null.
    if pd.notna(row.get('ParentAnswerListId')):
        parent_urls.append(CONCEPT_URL_PREFIX.format(row['ParentAnswerListId']))
    
    # Return pd.NA if the list is empty, otherwise return the list.
    return parent_urls if parent_urls else pd.NA

# Apply the function to the merged dataframe to create the new column.
concepts_with_hierarchy_df['parent_concept_urls'] = concepts_with_hierarchy_df.apply(create_parent_urls, axis=1)

# Display the updated dataframe with the new column.
# print(concepts_with_hierarchy_df[concepts_with_hierarchy_df['parent_concept_urls'].str.len() > 0][['id', 'parent_concept', 'ParentAnswerListId', 'parent_concept_urls']].head())

## Phase 5: UMLS Enhancement

Here, we'll query an external UMLS API to enrich the concepts with CUIs (Concept Unique Identifiers).

In [183]:
## Phase 5: LOINC to UMLS CUI Mapping

# This phase uses the SimpleLOINCMapper to map LOINC codes to UMLS CUIs.
# The UMLS API key is loaded from the config.json file.
# The code iterates through the LOINC codes in the `concepts_with_hierarchy_df` DataFrame and
# uses the mapper to find the corresponding UMLS CUI.
# The results are stored in a new DataFrame.

# 1. Load the UMLS API key from the config.json file
with open('UMLS_API_config.json', 'r') as f:
    config = json.load(f)
    api_key = config.get('api_key')
    rate_limit_delay = config.get('rate_limit_delay', 0.2) # Default to 0.2 if not specified

if not api_key:
    raise ValueError("UMLS API key not found in config.json")

# 2. Import the SimpleLOINCMapper class
from simple_loinc_mapper import SimpleLOINCMapper

# 3. Instantiate the mapper with the API key and rate limit
mapper = SimpleLOINCMapper(api_key=api_key, rate_limit=rate_limit_delay)

# 4. Filter the concepts_with_hierarchy_df DataFrame to exclude rows where concept_class is "Answer List"
# Assumes concepts_with_hierarchy_df has been loaded in a previous phase and has a 'concept_class' column
loinc_codes_all = concepts_with_hierarchy_df['id'].unique()
filtered_concepts_with_hierarchy_df = concepts_with_hierarchy_df[concepts_with_hierarchy_df['concept_class'] != 'Answer List']
loinc_codes_to_map = filtered_concepts_with_hierarchy_df['id'].unique()

print(f"Total unique LOINC codes before filtering: {len(loinc_codes_all)}")
print(f"Total unique LOINC codes to map (excluding 'Answer List'): {len(loinc_codes_to_map)}")
print(f"Number of LOINC codes excluded from mapping: {len(loinc_codes_all) - len(loinc_codes_to_map)}")

# 5. Create a list to hold the mapping results
loinc_to_cui_mappings = []

# 6. Iterate through the filtered LOINC codes and perform the mapping
for loinc_code in loinc_codes_to_map:
    # Get the CUI and additional info from the mapper
    result = mapper.search_loinc_code(loinc_code)
    if result:
        loinc_to_cui_mappings.append({
            'loinc_code': loinc_code,
            'cui': result.get('cui'),
            'cui_name': result.get('cui_name'),
            'source_ui': result.get('source_ui'),
            'source_name': result.get('source_name'),
            'mapping_method': result.get('mapping_method')
        })

# 7. Create a new DataFrame from the mapping results
loinc_cui_df = pd.DataFrame(loinc_to_cui_mappings)

# 8. Display the first few rows of the new DataFrame
# print("\nLOINC to UMLS CUI Mappings:")
# print(loinc_cui_df.head())

# 9. Merge with the original concepts_with_hierarchy_df and report mapping statistics
# Keep only the 'loinc_code' and 'cui' columns from loinc_cui_df for the merge
loinc_cui_for_merge = loinc_cui_df[['loinc_code', 'cui']]
merged_loinc_cui_df = pd.merge(concepts_with_hierarchy_df, loinc_cui_for_merge, left_on='id', right_on='loinc_code', how='left')
merged_loinc_cui_df.drop(columns=['loinc_code','ParentAnswerListId','parent_concept','match-id'], inplace=True)  # Drop unnecessary columns after merging
merged_loinc_cui_df.rename(columns={'cui': 'external_id'}, inplace=True)
merged_loinc_cui_df['extras.UMLS CUI'] = merged_loinc_cui_df['external_id']

# Count the number of rows with and without a matched CUI
matched_cui_count = merged_loinc_cui_df['external_id'].notna().sum()
unmatched_cui_count = merged_loinc_cui_df['external_id'].isna().sum()

print("\n--- Mapping Statistics ---")
print(f"Total rows in merged dataframe: {len(merged_loinc_cui_df)}")
print(f"Number of rows with a matched CUI: {matched_cui_count}")
print(f"Number of rows without a matched CUI: {unmatched_cui_count}")

# 10. Optionally, save the results to a CSV file
# output_path = os.path.join(output_folder, 'loinc_to_cui_mappings.csv')
# loinc_cui_df.to_csv(output_path, index=False)
# print(f"\nSaved mappings to {output_path}")

2025-08-14 12:20:06,658 - INFO - Loaded 465 mappings from local cache: loinc_cui_cache_2025-08-14.csv
Total unique LOINC codes before filtering: 511
Total unique LOINC codes to map (excluding 'Answer List'): 465
Number of LOINC codes excluded from mapping: 46

--- Mapping Statistics ---
Total rows in merged dataframe: 511
Number of rows with a matched CUI: 465
Number of rows without a matched CUI: 46


## Phase 6: Output Generation

The final phase is to save all the created OCL objects to a JSON files (`.json`) in the Bulk Import JSON-lines-like format for import into the OCL system.

In [186]:
# Quick cleanup of dfs before output

# Concepts - change type of 'extras.VersionFirstReleased' to string
merged_loinc_cui_df['extras.VersionLastChanged'] = merged_loinc_cui_df['extras.VersionLastChanged'].astype(str).replace('nan', '')

# Mappings - remove bad values:
# "extras": {"Sequence": null, "Required": null, "Cardinality": null, "Answer List Override": NaN, "Answer List Type Override": NaN}
# "extras": {"Answer List ID": null, "Answer List Type": null, "Sequence": null, "Score": null, "Local Answer Code": null}
# "extras": {"COMMENT": NaN}
for mapping in all_mappings:
    if "extras" in mapping and mapping["extras"]:
        # Create a list of keys to remove to avoid modifying the dictionary while iterating
        keys_to_remove = [
            key for key, value in mapping["extras"].items()
            if value is None or (isinstance(value, float) and pd.isna(value))
        ]

        for key in keys_to_remove:
            del mapping["extras"][key]


# FINAL VALIDATION - all concepts have a parent_concept_urls
print("--- Starting Final Validation ---")
# Check if any concept is missing parent_concept_urls
missing_parent_concepts = merged_loinc_cui_df[merged_loinc_cui_df['parent_concept_urls'].isnull()]
if not missing_parent_concepts.empty:
    print("ERROR: The following concepts are missing a parent_concept_url:")
    print(missing_parent_concepts[['id']]) # Adjust column names as needed
else:
    print("SUCCESS: All concepts have a parent_concept_url.")
print("--- Final Validation Complete ---")

--- Starting Final Validation ---
ERROR: The following concepts are missing a parent_concept_url:
           id
0    100000-9
1    100001-7
2    100002-5
3    100003-3
4    100004-1
..        ...
301  LL1038-0
302  LL1039-8
303  LL1040-6
304   LL104-1
305  LL1041-4

[296 rows x 1 columns]
--- Final Validation Complete ---


In [188]:
# Phase 6 code here

output_folder_path = 'output'
os.makedirs(output_folder_path, exist_ok=True)

CHUNK_SIZE = 50000

def is_valid_value(value):
    """
    Checks if a value is not NaN, None, an empty list/dict, or an empty string.
    Handles non-scalar values (like lists and pandas Series) safely.
    """
    if value is None:
        return False
    
    # Handle empty string specifically
    if isinstance(value, str) and value.strip() == "":
        return False

    # Handle scalar values first
    if not isinstance(value, (list, tuple, pd.Series, np.ndarray)):
        return not pd.isna(value)
    
    # Handle non-scalar values (lists, Series, etc.)
    # Consider them valid only if they contain at least one non-NaN value
    if isinstance(value, pd.Series):
        return not value.dropna().empty
    elif isinstance(value, (list, np.ndarray)):
        # Check for empty list/array or list/array of NaNs
        return any(pd.notna(v) for v in value)
    
    # For other types, check if they are "empty"
    if hasattr(value, '__len__') and len(value) == 0:
        return False

    return True

def write_chunks(data_list, file_prefix):
    """
    Chunks a list of dictionaries and writes each chunk to a separate JSON Lines file.
    """
    print(f"Writing {len(data_list)} records for '{file_prefix}' to chunked files...")
    for i in range(0, len(data_list), CHUNK_SIZE):
        chunk = data_list[i:i + CHUNK_SIZE]
        chunk_number = i // CHUNK_SIZE + 1
        output_file_name = f"{file_prefix}_{chunk_number}.json"
        output_file_path = os.path.join(output_folder_path, output_file_name)
        
        print(f"Writing chunk {chunk_number} of {len(data_list)//CHUNK_SIZE + 1} to {output_file_path}...")
        with open(output_file_path, 'w') as f:
            for record in chunk:
                json.dump(record, f)
                f.write('\n')


# === New cleanup step for merged_loinc_cui_df ===
# Iterate over columns starting with 'extras.' and replace bad values with None
for col in merged_loinc_cui_df.columns:
    if col.startswith('extras.'):
        merged_loinc_cui_df[col] = merged_loinc_cui_df[col].apply(lambda x: None if pd.isna(x) else x)

# Process the merged_loinc_cui_df for Concepts
concepts_data = []
print("Generating OCL Concepts from merged_loinc_cui_df...")
total_concepts = len(merged_loinc_cui_df)
for idx, row in merged_loinc_cui_df.iterrows():
    if (idx + 1) % 1000 == 0 or (idx + 1) == total_concepts:
        print(f"    Processed {idx + 1}/{total_concepts} concepts.")
    
    row_dict = row.to_dict()
    
    # Logic to dynamically process all 'names' attributes with locale_preferred flag
    names_list = []
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('names.'):
            if is_valid_value(value):
                # Use regex to extract name_type and locale
                match = re.match(r'names\.([^.]+)\.([^\[]+)\[\d+\]', key)
                if match:
                    name_type, locale = match.groups()
                    locale_preferred = (name_type == 'Fully-Specified')
                    names_list.append({'name': value, 'name_type': name_type, 'locale': locale, 'locale_preferred': locale_preferred})
            del row_dict[key]
    
    if names_list:
        row_dict['names'] = names_list
    
    # The 'description' is also in a specific format in the OCL
    description = row_dict.pop('description', None)
    if is_valid_value(description):
        row_dict['descriptions'] = [{'description': description, 'locale': 'en', 'description_type': 'Full'}]
    else:
        row_dict['descriptions'] = []

    # Logic to handle the 'extras' attribute as a single dictionary at the end
    extras_dict = {}
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('extras.'):
            if is_valid_value(value):
                # Clean the key name by removing 'extras.' prefix
                new_key = key[len('extras.'):]
                extras_dict[new_key] = value
            del row_dict[key]
    
    if extras_dict:
        row_dict['extras'] = extras_dict

    # Final cleanup of any remaining invalid values
    # UPDATED: The condition below is what was modified.
    cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
    if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
        del cleaned_row_dict['extras']

    concepts_data.append(cleaned_row_dict)

write_chunks(concepts_data, 'concepts')

# Process the all_mappings for Mappings
mappings_data = []
print("Generating OCL Mappings from all_mappings...")
total_mappings = len(all_mappings)

all_mappings_df = pd.DataFrame(all_mappings)

# === New cleanup step for all_mappings_df ===
for col in all_mappings_df.columns:
    if col.startswith('extras.'):
        all_mappings_df[col] = all_mappings_df[col].apply(lambda x: None if pd.isna(x) else x)

for idx, row in all_mappings_df.iterrows():
    if (idx + 1) % 1000 == 0 or (idx + 1) == total_mappings:
        print(f"    Processed {idx + 1}/{total_mappings} mappings.")
    
    row_dict = row.to_dict()

    # Logic to handle the 'extras' attribute as a single dictionary at the end
    extras_dict = {}
    keys_to_remove = []
    for key, value in list(row_dict.items()):
        if key.startswith('extras.'):
            if is_valid_value(value):
                # Clean the key name by removing 'extras.' prefix
                new_key = key[len('extras.'):]
                extras_dict[new_key] = value
            del row_dict[key]
    
    if extras_dict:
        row_dict['extras'] = extras_dict

    # Final cleanup of any remaining invalid values
    # UPDATED: The condition below is what was modified.
    cleaned_row_dict = {k: v for k, v in row_dict.items() if is_valid_value(v)}
    if 'extras' in cleaned_row_dict and not cleaned_row_dict['extras']:
        del cleaned_row_dict['extras']
        
    mappings_data.append(cleaned_row_dict)

write_chunks(mappings_data, 'mappings')

print("OCL Bulk Import files generated successfully in separate, chunked files.")

Generating OCL Concepts from merged_loinc_cui_df...
    Processed 511/511 concepts.
Writing 511 records for 'concepts' to chunked files...
Writing chunk 1 of 1 to output\concepts_1.json...
Generating OCL Mappings from all_mappings...
    Processed 1000/2381 mappings.
    Processed 2000/2381 mappings.
    Processed 2381/2381 mappings.
Writing 2381 records for 'mappings' to chunked files...
Writing chunk 1 of 1 to output\mappings_1.json...
OCL Bulk Import files generated successfully in separate, chunked files.


In [ ]:
# Final validation - all concepts have a parent_concept_url, no duplicate concepts or mappings
# Cleanup - VersionFirstReleased is a number if possible, or a string if numeric doesn't work